In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def make_student_success(n=600, seed=1):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame({
        "hours": rng.integers(0, 12, n),
        "attendance": rng.integers(40, 100, n),
        "sleep": rng.integers(4, 9, n),
        "practice_tests": rng.integers(0, 6, n),
    })
    y = (((X["hours"] >= 5) &
          (X["attendance"] >= 70) &
          (X["sleep"] >= 6))
          |
          (X["practice_tests"] >= 4)).astype(int)
    # Add noise (real-life mistakes)
    y = np.where(rng.random(n) < 0.07, 1-y, y)
    return X, y

# Create dataset

X, y = make_student_success()

#show the raw data 

print(X.head())
print(y[:5])

seeds = [1, 2, 3, 4, 5, 10, 20, 30]
scores = []

# train and test Split
for s in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=s, stratify=y
    )

#train the model
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    scores.append(acc)
    print("seed", s, "accuracy", round(acc, 3))

print("\nAvg accuracy:", round(float(np.mean(scores)), 3))
print("Min accuracy:", round(float(np.min(scores)), 3))
print("Max accuracy:", round(float(np.max(scores)), 3))

# Predict New Students
new_student = [[6, 80, 7, 3]]

print("Prediction : " , model.predict(new_student)[0])

# Repeating Train/Test Splits with Different Seeds

## What’s Happening?

We are splitting the **same dataset multiple times** using different random seeds.

Each seed creates a **different train/test split**.

Example idea:

Seed 1 → different split  
Seed 2 → different split  
Seed 3 → different split  
Seed 4 → different split  

Even though the **dataset is identical**, the **rows selected for training and testing change**.

This means the model is tested under **multiple conditions**.

---

# Why Do This?

Because **one single split can lie to you**.

Imagine this scenario:

Seed 1 → lucky split → accuracy = 0.91  
Seed 2 → unlucky split → accuracy = 0.82  

Which one is correct?

**Answer: Neither.**

The **real performance** of the model is the **average across many splits**.

This is a **step toward robust evaluation**.

You are **stress-testing the model** instead of trusting one lucky experiment.

This is a **professional machine learning habit**.

---